# Event Impact / Turning-Point Analysis
Single-creator (Camihawke) interrupted-time-series study.
This notebook implements the methodology in `Ali/Archive/plans/Archive/plans/EVENT_IMPACT_PLAN.md`.
Run top-to-bottom; all outputs go to `Ali/outputs/event_impact/`.

In [1]:
import pathlib
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

ROOT   = pathlib.Path('d:/Polythecninco di Milano/AFB_Lab')
OUTDIR = ROOT / 'Ali/outputs/event_impact'
OUTDIR.mkdir(parents=True, exist_ok=True)
print('Output dir:', OUTDIR)

Output dir: d:\Polythecninco di Milano\AFB_Lab\Ali\outputs\event_impact


## Proxy 1 — C1 monthly cluster matrix (primary proxy)

In [2]:
MATRIX_PATH = ROOT / 'Mickey/RFM_IG_rolling_quadrimesters/Ig_RCEDTG_k4_monthly_cluster_matrix.csv'
mat = pd.read_csv(MATRIX_PATH, low_memory=False)
print('matrix shape:', mat.shape)
print('matrix dtypes (first 10 cols):')
print(mat.dtypes.head(10))
print('\nfirst 3 rows (id cols + first 5 month cols):')
id_cols = ['user_key', 'user_id', 'username']
id_cols_present = [c for c in id_cols if c in mat.columns]
month_cols = [c for c in mat.columns if c not in id_cols]
print(mat[id_cols_present + month_cols[:5]].head(3))

matrix shape: (18752, 42)
matrix dtypes (first 10 cols):
user_key     object
user_id     float64
username     object
2023-01      object
2023-02      object
2023-03      object
2023-04      object
2023-05      object
2023-06      object
2023-07      object
dtype: object

first 3 rows (id cols + first 5 month cols):
           user_key       user_id   username         2023-01         2023-02  \
0  1452263009584630  1.452263e+15       00bc  Not yet active  Not yet active   
1  1529027545186143  1.529028e+15  074serena  Not yet active  Not yet active   
2  4117121325206183  4.117121e+15  077.matte  Not yet active  Not yet active   

          2023-03         2023-04         2023-05  
0  Not yet active  Not yet active  Not yet active  
1  Not yet active  Not yet active  Not yet active  
2  Not yet active  Not yet active  Not yet active  


In [3]:
# Melt wide -> long
long = mat.melt(
    id_vars=id_cols_present,
    value_vars=month_cols,
    var_name='month_str',
    value_name='cluster'
)
# Parse month column (expected format YYYY-MM)
long['month'] = pd.PeriodIndex(long['month_str'], freq='M')
print('long shape:', long.shape)
print('cluster value counts (sample):')
print(long['cluster'].value_counts().head(10))

long shape: (731328, 6)
cluster value counts (sample):
cluster
Inactive          448639
Not yet active    117359
C3                109819
C1                 31302
C4                 12290
C2                 11919
Name: count, dtype: int64


In [4]:
ACTIVE_CLUSTERS = {'C1', 'C2', 'C3', 'C4'}
INACTIVE_LABELS = {'Inactive', 'Not yet active'}

def build_c1_share(df):
    """
    Returns monthly DataFrame with:
      n_C1, n_active (excl Inactive/NYA), n_all_users,
      c1_share_active  = n_C1 / n_active  (denominator 1: active that month)
      c1_share_total   = n_C1 / n_all_users  (denominator 2: all users ever in matrix)
      c1_to_c4         = n_C1 / max(1, n_C4)
    """
    total_users = df['user_id'].nunique() if 'user_id' in df.columns else len(df) // len(month_cols)
    rows = []
    for month, grp in df.groupby('month'):
        n_C1     = (grp['cluster'] == 'C1').sum()
        n_C4     = (grp['cluster'] == 'C4').sum()
        n_active = grp['cluster'].isin(ACTIVE_CLUSTERS).sum()
        rows.append({
            'month': month,
            'n_C1': int(n_C1),
            'n_active': int(n_active),
            'n_C4': int(n_C4),
            'c1_share_active': n_C1 / n_active if n_active > 0 else np.nan,
            'c1_share_total': n_C1 / total_users,
            'c1_to_c4': n_C1 / max(1, n_C4),
        })
    return pd.DataFrame(rows).sort_values('month').reset_index(drop=True)

c1_series = build_c1_share(long)
print('c1_series shape:', c1_series.shape)
print('Date range (Proxy 1):', c1_series['month'].min(), '->', c1_series['month'].max())
print()
print('=== C1 SHARE MONTHLY SERIES ===')
print(c1_series.to_string(index=False))

c1_series shape: (39, 7)
Date range (Proxy 1): 2023-01 -> 2026-03

=== C1 SHARE MONTHLY SERIES ===
  month  n_C1  n_active  n_C4  c1_share_active  c1_share_total  c1_to_c4
2023-01  1128      6020   506         0.187375        0.060282  2.229249
2023-02  1012      5814   438         0.174063        0.054083  2.310502
2023-03   768      5355   443         0.143417        0.041043  1.733634
2023-04  1192      5380   282         0.221561        0.063702  4.226950
2023-05  1345      6001   300         0.224129        0.071879  4.483333
2023-06  1209      5824   342         0.207589        0.064611  3.535088
2023-07  1615      7048   455         0.229143        0.086308  3.549451
2023-08  1562      7285   554         0.214413        0.083476  2.819495
2023-09  1347      6375   364         0.211294        0.071986  3.700549
2023-10   934      5809   402         0.160785        0.049914  2.323383
2023-11   645      4335   216         0.148789        0.034470  2.986111
2023-12   488      3178  

## Proxy 2 — Sentiment monthly series

In [5]:
SENT_PATH = ROOT / 'Ali/outputs/stage2_sentiment/sentiment_instagram.parquet'
sent = pd.read_parquet(SENT_PATH)
print('sentiment shape:', sent.shape)
print('sentiment dtypes:')
print(sent.dtypes)
print('\nfirst 3 rows:')
print(sent.head(3))

sentiment shape: (487604, 33)
sentiment dtypes:
comment_id               object
media_id                 object
media_context              bool
sentiment                object
sentiment_score         float64
emotion                  object
intensity                object
sarcasm                    bool
toxicity                 object
intent                   object
target                   object
lang                     object
sentiment_cat            object
sarcasm_label            object
author_id                object
platform                 object
text_length               int64
word_count                int64
emoji_count               int64
unique_emoji_count      float64
emoji_entropy           float64
emoji_variety_ratio     float64
emoji_per_word_ratio    float64
url_count                 int64
mention_count             int64
hashtag_count             int64
exclamation_count         int64
question_count            int64
avg_word_length         float64
has_numbers             

In [6]:
# Parse timestamp - detect unit (s vs ms)
ts = sent['timestamp']
print('timestamp dtype:', ts.dtype, '| sample values:', ts.dropna().iloc[:5].tolist())

if pd.api.types.is_numeric_dtype(ts):
    sample_val = ts.dropna().iloc[0]
    if sample_val > 1e12:   # milliseconds
        unit = 'ms'
    else:                   # seconds
        unit = 's'
    print(f'Detected timestamp unit: {unit}')
    sent['dt'] = pd.to_datetime(ts, unit=unit, utc=True)
else:
    sent['dt'] = pd.to_datetime(ts, utc=True)
    unit = 'already datetime'
    print('Timestamp already datetime-like')

sent['month'] = sent['dt'].dt.to_period('M')
print('\nSentiment date range:', sent['month'].min(), '->', sent['month'].max())

timestamp dtype: object | sample values: ['2020-10-28 12:18:12', '2020-10-28 12:18:06', '2020-10-28 12:18:00', '2020-10-28 12:17:54', '2020-10-28 12:20:08']
Timestamp already datetime-like

Sentiment date range: 2016-08 -> 2026-03


In [7]:
TOX_MAP = {'none': 0, 'mild': 1, 'severe': 2}

def build_sentiment_series(df):
    # toxicity is categorical str -> map to numeric for mean
    if 'toxicity' in df.columns:
        df = df.copy()
        df['toxicity_num'] = df['toxicity'].map(TOX_MAP)
    grps = df.groupby('month')
    rows = []
    for month, g in grps:
        n = len(g)
        rows.append({
            'month': month,
            'n_comments': n,
            'mean_sentiment': g['sentiment_score'].mean() if 'sentiment_score' in g.columns else np.nan,
            'pct_positive': (g['sentiment_cat'] == 'positive').sum() / n if 'sentiment_cat' in g.columns else np.nan,
            'pct_negative': (g['sentiment_cat'] == 'negative').sum() / n if 'sentiment_cat' in g.columns else np.nan,
            'sarcasm_rate': g['sarcasm'].astype(float).mean() if 'sarcasm' in g.columns else np.nan,
            'mean_toxicity': g['toxicity_num'].mean() if 'toxicity_num' in g.columns else np.nan,
            'pct_toxic_any': (g['toxicity_num'] > 0).mean() if 'toxicity_num' in g.columns else np.nan,
        })
    return pd.DataFrame(rows).sort_values('month').reset_index(drop=True)

sent_series = build_sentiment_series(sent)
print('sent_series shape:', sent_series.shape)
print('\n=== SENTIMENT MONTHLY SERIES (head 12) ===')
print(sent_series.head(12).to_string(index=False))

sent_series shape: (116, 8)

=== SENTIMENT MONTHLY SERIES (head 12) ===
  month  n_comments  mean_sentiment  pct_positive  pct_negative  sarcasm_rate  mean_toxicity  pct_toxic_any
2016-08         258        0.517054      0.771318      0.027132      0.054264       0.011628       0.011628
2016-09         623        0.483868      0.703050      0.024077      0.030498       0.012841       0.012841
2016-10         819        0.439683      0.702076      0.080586      0.014652       0.009768       0.007326
2016-11        1159        0.534254      0.790336      0.019845      0.045729       0.004314       0.004314
2016-12         892        0.495908      0.774664      0.034753      0.035874       0.005605       0.005605
2017-01         671        0.476602      0.737705      0.035768      0.010432       0.011923       0.011923
2017-02        1040        0.535029      0.776923      0.028846      0.024038       0.006731       0.005769
2017-03         887        0.526043      0.799324      0.024803 

## Proxy 3 — Persona composition over time

In [8]:
PERSONA_PATH  = ROOT / 'Ali/outputs/stage2_persona_combined/user_personas_combined.parquet'
COMMENTS_PATH = ROOT / 'Ali/outputs/comments_ml.parquet'

personas = pd.read_parquet(PERSONA_PATH)
print('personas shape:', personas.shape)
print('personas dtypes:')
print(personas.dtypes)
print('persona_codename value counts:')
print(personas['persona_codename'].value_counts())

personas shape: (40019, 10)
personas dtypes:
author_id                 object
total_comments             int64
activity_span_days         int64
mean_hours_to_comment    float64
pct_comments_under_1h    float64
reply_ratio              float64
mean_word_count          float64
persona_codename          object
confidence               float64
justification             object
dtype: object
persona_codename value counts:
persona_codename
THE_TAGGER                 11628
THE_CASUAL_COMPLIMENTER     7802
THE_EMOJI_REACTOR           5764
THE_STORYTELLER             4471
THE_SUPERFAN                4191
THE_INQUIRER                2608
THE_CRITIC                  2253
THE_ADVISOR                  865
THE_SPAMMER                  218
THE_HATER                    164
Name: count, dtype: int64


In [9]:
comments = pd.read_parquet(COMMENTS_PATH)
print('comments_ml shape:', comments.shape)
print('comments_ml dtypes:')
print(comments.dtypes)
print('\nsample author_id values:', comments['author_id'].dropna().iloc[:3].tolist() if 'author_id' in comments.columns else 'NO author_id')

comments_ml shape: (3661495, 20)
comments_ml dtypes:
comment_id               object
author_id                object
media_id                 object
platform                 object
text_length               int64
word_count                int64
emoji_count               int64
unique_emoji_count      float64
emoji_entropy           float64
emoji_variety_ratio     float64
emoji_per_word_ratio    float64
url_count                 int64
mention_count             int64
hashtag_count             int64
exclamation_count         int64
question_count            int64
avg_word_length         float64
has_numbers               int64
has_links                 int64
timestamp                object
dtype: object

sample author_id values: ['3482058005293604', '3103734239813346', '908266745384508']


In [10]:
# Parse comments timestamp
cts = comments['timestamp']
print('comments timestamp dtype:', cts.dtype, '| sample:', cts.dropna().iloc[:5].tolist())

if pd.api.types.is_numeric_dtype(cts):
    sv = cts.dropna().iloc[0]
    cu = 'ms' if sv > 1e12 else 's'
    print(f'Detected unit: {cu}')
    comments['dt'] = pd.to_datetime(cts, unit=cu, utc=True)
else:
    # Mixed ISO8601 (some with +00:00 offset, some without) -> use ISO8601 parser
    comments['dt'] = pd.to_datetime(cts, format='ISO8601', utc=True)
    cu = 'ISO8601 string'
    print(f'Parsed as ISO8601 strings, utc=True')

comments['month'] = comments['dt'].dt.to_period('M')
print('comments date range:', comments['month'].min(), '->', comments['month'].max())

# Join persona onto comments
personas['author_id'] = personas['author_id'].astype(str)
comments['author_id'] = comments['author_id'].astype(str)

merged = comments.merge(personas[['author_id', 'persona_codename']], on='author_id', how='left')
match_rate = merged['persona_codename'].notna().mean()
print(f'\nPersona join match rate: {match_rate:.1%} of {len(merged)} comments')
print('Unmatched (no persona label):', merged['persona_codename'].isna().sum())

comments timestamp dtype: object | sample: ['2026-03-19 14:43:02', '2026-03-19 13:39:14', '2026-03-19 13:39:00', '2026-03-19 13:55:56', '2026-03-19 21:29:53']
Parsed as ISO8601 strings, utc=True
comments date range: 2008-02 -> 2026-03



Persona join match rate: 2.6% of 3661495 comments
Unmatched (no persona label): 3564904


In [11]:
def build_persona_series(df):
    rows = []
    for month, g in df.groupby('month'):
        n = len(g)
        n_labelled = g['persona_codename'].notna().sum()
        base = max(1, n_labelled)
        row = {'month': month, 'n_comments': n, 'n_labelled': n_labelled}
        for persona in df['persona_codename'].dropna().unique():
            row[f'pct_{persona}'] = (g['persona_codename'] == persona).sum() / base
        rows.append(row)
    out = pd.DataFrame(rows).sort_values('month').reset_index(drop=True)
    return out

persona_series = build_persona_series(merged)
print('persona_series shape:', persona_series.shape)
print('Date range (Proxy 3):', persona_series['month'].min(), '->', persona_series['month'].max())

# Find SUPERFAN/CRITIC/HATER cols
sf_cols  = [c for c in persona_series.columns if 'SUPERFAN'  in c]
crit_cols = [c for c in persona_series.columns if 'CRITIC'   in c or 'HATER' in c]
print('\nSUPERFAN cols:', sf_cols)
print('CRITIC/HATER cols:', crit_cols)
print('\n=== PERSONA SERIES (head 6, key cols) ===')
show_cols = ['month', 'n_comments'] + sf_cols[:1] + crit_cols[:2]
print(persona_series[show_cols].head(6).to_string(index=False))

persona_series shape: (206, 13)
Date range (Proxy 3): 2008-02 -> 2026-03

SUPERFAN cols: ['pct_THE_SUPERFAN']
CRITIC/HATER cols: ['pct_THE_CRITIC', 'pct_THE_HATER']

=== PERSONA SERIES (head 6, key cols) ===
  month  n_comments  pct_THE_SUPERFAN  pct_THE_CRITIC  pct_THE_HATER
2008-02           2               0.0             0.0            0.0
2009-03          56               0.0             0.0            0.0
2009-04        1178               0.0             0.0            0.0
2009-05         755               0.0             0.0            0.0
2009-06        1099               0.0             0.0            0.0
2009-07        1313               0.0             0.0            0.0


## Events — load and compute date-coverage / testability table

In [12]:
EVENTS_PATH = ROOT / 'Ali/camihawke_major_events_timeline.csv'
events = pd.read_csv(EVENTS_PATH)
print('events shape:', events.shape)
print('events dtypes:', events.dtypes.to_dict())
print('\nAll events:')
print(events.to_string(index=False))

events shape: (17, 5)
events dtypes: {'Date': dtype('O'), 'Event_Title': dtype('O'), 'Category': dtype('O'), 'Details': dtype('O'), 'Metrics_or_Scope': dtype('O')}

All events:
      Date                                                              Event_Title                    Category                                                                                                                                                                             Details                                   Metrics_or_Scope
 11/1/2016                                      Initial Digital Footprint Expansion      Professional Milestone                 Began active distribution of curated comedic, parodic, and narrative video content across Snapchat, Facebook, and Instagram under the artistic moniker 'Camihawke'.                        Multi-platform media launch
 9/15/2017                                   Rai Radio2 On-Air Broadcast Engagement      Professional Milestone                          

In [13]:
# Parse Date column (expected US format M/D/YYYY)
events['event_date'] = pd.to_datetime(events['Date'], dayfirst=False, infer_datetime_format=True)
events['event_month'] = events['event_date'].dt.to_period('M')
print('Parsed event months:')
print(events[['Date', 'event_date', 'event_month', 'Event_Title', 'Category']].to_string(index=False))

Parsed event months:
      Date event_date event_month                                                              Event_Title                    Category
 11/1/2016 2016-11-01     2016-11                                      Initial Digital Footprint Expansion      Professional Milestone
 9/15/2017 2017-09-15     2017-09                                   Rai Radio2 On-Air Broadcast Engagement      Professional Milestone
 9/16/2019 2019-09-16     2019-09                                  Co-Hosting Debut on National Television      Professional Milestone
 11/5/2019 2019-11-05     2019-11 Macchianera Internet Award & Corporate Social Responsibility Appointment     Accolade / Philanthropy
12/14/2019 2019-12-14     2019-12                                        TEDxRimini Technical Presentation      Professional Milestone
 4/20/2021 2021-04-20     2021-04                                                  Debut Novel Publication      Professional Milestone
 4/15/2023 2023-04-15     2023-04 

In [14]:
# Proxy windows
p1_min, p1_max = c1_series['month'].min(),  c1_series['month'].max()
p2_min, p2_max = sent_series['month'].min(), sent_series['month'].max()
p3_min, p3_max = persona_series['month'].min(), persona_series['month'].max()

WIN = 3  # default window half-width in months

def testable(event_m, proxy_min, proxy_max, win=WIN):
    """Returns True if event_m has >= win months of data on EACH side (excl event month)."""
    pre_start  = event_m - win
    post_end   = event_m + win
    return pre_start >= proxy_min and post_end <= proxy_max

rows = []
for _, ev in events.iterrows():
    em = ev['event_month']
    rows.append({
        'Event_Title': ev['Event_Title'],
        'Category': ev['Category'],
        'event_month': em,
        'P1_testable_win3': testable(em, p1_min, p1_max),
        'P2_testable_win3': testable(em, p2_min, p2_max),
        'P3_testable_win3': testable(em, p3_min, p3_max),
        'P1_in_window': p1_min <= em <= p1_max,
        'P2_in_window': p2_min <= em <= p2_max,
        'P3_in_window': p3_min <= em <= p3_max,
    })

coverage = pd.DataFrame(rows)

print(f'Proxy 1 (cluster):  {p1_min} -> {p1_max}')
print(f'Proxy 2 (sentiment):{p2_min} -> {p2_max}')
print(f'Proxy 3 (persona):  {p3_min} -> {p3_max}')
print(f'\nDefault window: +/-{WIN} months (excl event month)')
print('\n=== EVENT DATE-COVERAGE AND TESTABILITY TABLE ===')
print(coverage.to_string(index=False))

Proxy 1 (cluster):  2023-01 -> 2026-03
Proxy 2 (sentiment):2016-08 -> 2026-03
Proxy 3 (persona):  2008-02 -> 2026-03

Default window: +/-3 months (excl event month)

=== EVENT DATE-COVERAGE AND TESTABILITY TABLE ===
                                                             Event_Title                    Category event_month  P1_testable_win3  P2_testable_win3  P3_testable_win3  P1_in_window  P2_in_window  P3_in_window
                                     Initial Digital Footprint Expansion      Professional Milestone     2016-11             False              True              True         False          True          True
                                  Rai Radio2 On-Air Broadcast Engagement      Professional Milestone     2017-09             False              True              True         False          True          True
                                 Co-Hosting Debut on National Television      Professional Milestone     2019-09             False              True         

In [15]:
# Flag overlapping windows: for each pair of testable events, check if their +/-WIN windows overlap
testable_p1 = coverage[coverage['P1_testable_win3']]['event_month'].tolist()

def period_gap_months(a, b):
    """Absolute gap in months between two Period('M') objects."""
    return abs(a.ordinal - b.ordinal)

overlap_flags = {}
for i, em in enumerate(testable_p1):
    overlaps_with = []
    for j, other in enumerate(testable_p1):
        if i == j:
            continue
        # windows overlap if gap <= 2*WIN (i.e. the +WIN post of one reaches the -WIN pre of the other)
        if period_gap_months(em, other) <= 2 * WIN:
            overlaps_with.append(str(other))
    overlap_flags[str(em)] = ', '.join(overlaps_with) if overlaps_with else ''

coverage['P1_overlap_with'] = coverage['event_month'].astype(str).map(overlap_flags).fillna('')
print('\n=== OVERLAP FLAGS (P1, win=3) ===')
print(coverage[['Event_Title', 'event_month', 'P1_testable_win3', 'P1_overlap_with']].to_string(index=False))


=== OVERLAP FLAGS (P1, win=3) ===
                                                             Event_Title event_month  P1_testable_win3                             P1_overlap_with
                                     Initial Digital Footprint Expansion     2016-11             False                                            
                                  Rai Radio2 On-Air Broadcast Engagement     2017-09             False                                            
                                 Co-Hosting Debut on National Television     2019-09             False                                            
Macchianera Internet Award & Corporate Social Responsibility Appointment     2019-11             False                                            
                                       TEDxRimini Technical Presentation     2019-12             False                                            
                                                 Debut Novel Publication     2021-0

In [16]:
# Save monthly series combined
# Convert Period to string for parquet compatibility
c1_out   = c1_series.copy();   c1_out['month']   = c1_out['month'].astype(str)
sent_out = sent_series.copy(); sent_out['month']  = sent_out['month'].astype(str)
pers_out = persona_series.copy(); pers_out['month'] = pers_out['month'].astype(str)

monthly = c1_out.merge(sent_out, on='month', how='outer', suffixes=('_p1', '_p2'))
monthly = monthly.merge(pers_out[['month'] + sf_cols + crit_cols], on='month', how='outer')
monthly = monthly.sort_values('month').reset_index(drop=True)

monthly.to_parquet(OUTDIR / 'monthly_series.parquet', index=False)
print('Saved monthly_series.parquet, shape:', monthly.shape)
print(monthly.columns.tolist())

coverage.to_csv(OUTDIR / 'event_coverage.csv', index=False)
print('Saved event_coverage.csv')

Saved monthly_series.parquet, shape: (206, 17)
['month', 'n_C1', 'n_active', 'n_C4', 'c1_share_active', 'c1_share_total', 'c1_to_c4', 'n_comments', 'mean_sentiment', 'pct_positive', 'pct_negative', 'sarcasm_rate', 'mean_toxicity', 'pct_toxic_any', 'pct_THE_SUPERFAN', 'pct_THE_CRITIC', 'pct_THE_HATER']
Saved event_coverage.csv


## Test 1 — Pre/Post window comparison (Mann-Whitney U + Cliff's delta, BH-FDR corrected)

In [17]:
from scipy import stats as sp_stats

# -----------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------

def cliff_delta(a, b):
    """Cliff's delta: (proportion of a>b pairs) - (proportion of a<b pairs)."""
    a, b = np.asarray(a), np.asarray(b)
    n = len(a) * len(b)
    if n == 0:
        return np.nan
    more = np.sum(a[:, None] > b[None, :])
    less = np.sum(a[:, None] < b[None, :])
    return (more - less) / n

def cliff_magnitude(d):
    ad = abs(d)
    if ad < 0.147:  return 'negligible'
    if ad < 0.330:  return 'small'
    if ad < 0.474:  return 'medium'
    return 'large'

def get_window(series, event_m, win, val_col):
    """
    Extract pre and post arrays for a monthly series.
    series must have a 'month' column (Period or string) and val_col.
    Excludes the event month itself.
    Returns (pre_vals, post_vals) as numpy arrays.
    """
    s = series.copy()
    if not pd.api.types.is_period_dtype(s['month']):
        s['month'] = pd.PeriodIndex(s['month'], freq='M')
    pre  = s[(s['month'] >= event_m - win) & (s['month'] < event_m)][val_col].dropna().values
    post = s[(s['month'] > event_m) & (s['month'] <= event_m + win)][val_col].dropna().values
    return pre, post

def run_prepost(series, event_m, win, val_col, proxy_name, event_title):
    """Run MW-U and t-test for one event x one proxy x one window."""
    pre, post = get_window(series, event_m, win, val_col)
    result = {
        'event_title': event_title,
        'proxy': proxy_name,
        'window': win,
        'event_month': str(event_m),
        'n_pre': len(pre),
        'n_post': len(post),
        'mean_pre': np.mean(pre) if len(pre) else np.nan,
        'mean_post': np.mean(post) if len(post) else np.nan,
        'diff_means': np.nan,
        'cliff_delta': np.nan,
        'cliff_mag': '',
        'mwu_stat': np.nan,
        'mwu_p': np.nan,
        'welch_p': np.nan,
        'testable': False,
        'raw_p': np.nan,
    }
    if len(pre) < 2 or len(post) < 2:
        return result
    result['testable'] = True
    result['diff_means'] = result['mean_post'] - result['mean_pre']
    cd = cliff_delta(post, pre)
    result['cliff_delta'] = cd
    result['cliff_mag'] = cliff_magnitude(cd)
    try:
        mwu = sp_stats.mannwhitneyu(pre, post, alternative='two-sided')
        result['mwu_stat'] = mwu.statistic
        result['mwu_p']    = mwu.pvalue
        result['raw_p']    = mwu.pvalue
    except Exception as e:
        result['mwu_p'] = np.nan
        result['raw_p'] = np.nan
    try:
        result['welch_p'] = sp_stats.ttest_ind(pre, post, equal_var=False).pvalue
    except Exception:
        pass
    return result

# -----------------------------------------------------------------------
# Build the event registry with testability info for each proxy series
# -----------------------------------------------------------------------

# Load monthly c1 series with Period index
c1_m = c1_series.copy()  # already has Period months

# Reload sent and persona series with Period index
sent_m = sent_series.copy()
sent_m['month'] = pd.PeriodIndex(sent_m['month'].astype(str), freq='M')

pers_m = persona_series.copy()
pers_m['month'] = pd.PeriodIndex(pers_m['month'].astype(str), freq='M')

# Events already parsed above; ensure event_month is Period
events_m = events.copy()
events_m['event_month'] = pd.PeriodIndex(events_m['event_month'].astype(str), freq='M')

# Proxy definitions: (series_df, value_column, proxy_label, min_period, max_period)
proxies = [
    (c1_m,   'c1_share_active', 'P1_c1_share_active', c1_m['month'].min(),   c1_m['month'].max()),
    (c1_m,   'c1_share_total',  'P1_c1_share_total',  c1_m['month'].min(),   c1_m['month'].max()),
    (c1_m,   'c1_to_c4',        'P1_c1_to_c4',        c1_m['month'].min(),   c1_m['month'].max()),
    (sent_m, 'mean_sentiment',  'P2_mean_sentiment',  sent_m['month'].min(), sent_m['month'].max()),
    (sent_m, 'pct_positive',    'P2_pct_positive',    sent_m['month'].min(), sent_m['month'].max()),
    (sent_m, 'pct_negative',    'P2_pct_negative',    sent_m['month'].min(), sent_m['month'].max()),
    (sent_m, 'mean_toxicity',   'P2_mean_toxicity',   sent_m['month'].min(), sent_m['month'].max()),
    (pers_m, 'pct_THE_SUPERFAN','P3_pct_superfan',    pers_m['month'].min(), pers_m['month'].max()),
    (pers_m, 'pct_THE_CRITIC',  'P3_pct_critic',      pers_m['month'].min(), pers_m['month'].max()),
    (pers_m, 'pct_THE_HATER',   'P3_pct_hater',       pers_m['month'].min(), pers_m['month'].max()),
]

WINDOWS = [2, 3, 6]

all_rows = []

for _, ev in events_m.iterrows():
    em = ev['event_month']
    title = ev['Event_Title'].strip()
    for (series_df, val_col, proxy_lbl, p_min, p_max) in proxies:
        for win in WINDOWS:
            # Check testability
            if not (em - win >= p_min and em + win <= p_max):
                all_rows.append({
                    'event_title': title,
                    'proxy': proxy_lbl,
                    'window': win,
                    'event_month': str(em),
                    'n_pre': 0, 'n_post': 0,
                    'mean_pre': np.nan, 'mean_post': np.nan,
                    'diff_means': np.nan, 'cliff_delta': np.nan, 'cliff_mag': '',
                    'mwu_stat': np.nan, 'mwu_p': np.nan, 'welch_p': np.nan,
                    'testable': False, 'raw_p': np.nan,
                })
                continue
            r = run_prepost(series_df, em, win, val_col, proxy_lbl, title)
            all_rows.append(r)

test1_raw = pd.DataFrame(all_rows)
print('Test 1 raw rows:', len(test1_raw))
print('Testable rows:', test1_raw['testable'].sum())
print('Untestable rows:', (~test1_raw['testable']).sum())

Test 1 raw rows: 510
Testable rows: 379
Untestable rows: 131


In [18]:
from statsmodels.stats.multitest import multipletests

# BH-FDR correction over the full family of testable rows
testable_mask = test1_raw['testable'] & test1_raw['raw_p'].notna()
test1_raw['fdr_q'] = np.nan

if testable_mask.sum() > 0:
    _, q_vals, _, _ = multipletests(
        test1_raw.loc[testable_mask, 'raw_p'].values,
        alpha=0.05, method='fdr_bh'
    )
    test1_raw.loc[testable_mask, 'fdr_q'] = q_vals

print(f'FDR correction applied over {testable_mask.sum()} testable comparisons.')
print(f'Significant after FDR (q<0.05): {(test1_raw["fdr_q"] < 0.05).sum()}')
print(f'Significant after FDR (q<0.10): {(test1_raw["fdr_q"] < 0.10).sum()}')

# Overlap flags from coverage table (load from CSV, keyed by event_month)
cov = pd.read_csv(OUTDIR / 'event_coverage.csv')
overlap_map = dict(zip(cov['event_month'], cov['P1_overlap_with']))
test1_raw['p1_overlap_with'] = test1_raw['event_month'].map(overlap_map).fillna('')

# Direction flag: was the post-change in the expected direction?
# Career/commercial -> expect c1_share UP, sentiment UP, superfan UP, critic/hater DOWN
# Personal/relationship -> ambiguous, test both
career_cats = {'Professional Milestone', 'Accolade / Philanthropy', 'Commercial Metric'}
personal_cats = {'Personal / Media Tracking', 'Personal / Public Relations'}

cat_map = dict(zip(events_m['Event_Title'].str.strip(), events_m['Category']))
test1_raw['category'] = test1_raw['event_title'].map(cat_map)

def expected_direction(row):
    """Returns +1 if higher post is expected, -1 if lower, 0 if ambiguous."""
    cat = row['category']
    proxy = row['proxy']
    if cat in career_cats:
        if proxy in ('P1_c1_share_active', 'P1_c1_share_total', 'P1_c1_to_c4',
                     'P2_mean_sentiment', 'P2_pct_positive', 'P3_pct_superfan'):
            return +1
        if proxy in ('P2_pct_negative', 'P2_mean_toxicity', 'P3_pct_critic', 'P3_pct_hater'):
            return -1
    return 0  # ambiguous (personal events or unknown)

test1_raw['expected_dir'] = test1_raw.apply(expected_direction, axis=1)

def direction_ok(row):
    if row['expected_dir'] == 0:
        return 'ambiguous'
    if pd.isna(row['cliff_delta']):
        return 'no_data'
    obs_dir = np.sign(row['cliff_delta'])   # cliff_delta = post-pre direction
    return 'yes' if obs_dir == row['expected_dir'] else 'no'

test1_raw['direction_ok'] = test1_raw.apply(direction_ok, axis=1)

# Print summary for win=3 only
w3 = test1_raw[(test1_raw['window'] == 3) & test1_raw['testable']].copy()
w3_show = w3[['event_title', 'event_month', 'proxy', 'mean_pre', 'mean_post',
              'diff_means', 'cliff_delta', 'cliff_mag',
              'mwu_p', 'fdr_q', 'direction_ok', 'p1_overlap_with']].copy()
w3_show = w3_show.sort_values(['event_month', 'proxy'])
print('\n=== TEST 1 RESULTS (window=3, testable only) ===')
pd.set_option('display.max_colwidth', 45)
pd.set_option('display.width', 200)
print(w3_show.to_string(index=False))

FDR correction applied over 379 testable comparisons.
Significant after FDR (q<0.05): 0
Significant after FDR (q<0.10): 0

=== TEST 1 RESULTS (window=3, testable only) ===
                                                             event_title event_month              proxy  mean_pre  mean_post  diff_means  cliff_delta  cliff_mag    mwu_p    fdr_q direction_ok                             p1_overlap_with
                                     Initial Digital Footprint Expansion     2016-11  P2_mean_sentiment  0.480202   0.502513    0.022311     0.333333     medium 0.700000 0.911684          yes                                            
                                     Initial Digital Footprint Expansion     2016-11   P2_mean_toxicity  0.011412   0.008086   -0.003326    -0.555556      large 0.400000 0.907784          yes                                            
                                     Initial Digital Footprint Expansion     2016-11    P2_pct_negative  0.043932   0.03

## Test 2 — Interrupted Time Series (ITS) with HAC standard errors

In [19]:
import statsmodels.api as sm

MIN_PRE_POST = 4  # require at least 4 months on each side

def run_its(series, event_m, val_col, proxy_name, event_title, min_pp=MIN_PRE_POST):
    """
    Fit segmented OLS:
      y = b0 + b1*time + b2*post + b3*(time*post) + e
    with Newey-West (HAC) SEs.
    b2 = level change, b3 = slope change.
    Returns dict with coefficients, HAC SEs, p-values.
    """
    s = series.copy()
    if not pd.api.types.is_period_dtype(s['month']):
        s['month'] = pd.PeriodIndex(s['month'].astype(str), freq='M')
    s = s.sort_values('month').reset_index(drop=True)
    s = s[s[val_col].notna()].copy()

    # Use all data in the series (not just a window), centred at event
    event_ord = event_m.ordinal
    s['time'] = s['month'].apply(lambda m: m.ordinal - event_ord)
    s['post'] = (s['month'] > event_m).astype(int)
    s['time_post'] = s['time'] * s['post']

    pre_n  = (s['post'] == 0).sum()
    post_n = (s['post'] == 1).sum()

    null = {
        'event_title': event_title, 'proxy': proxy_name,
        'event_month': str(event_m),
        'n_pre_its': pre_n, 'n_post_its': post_n,
        'its_b0': np.nan, 'its_b1': np.nan, 'its_b2': np.nan, 'its_b3': np.nan,
        'its_b2_se': np.nan, 'its_b3_se': np.nan,
        'its_b2_p': np.nan, 'its_b3_p': np.nan,
        'its_testable': False,
    }
    if pre_n < min_pp or post_n < min_pp:
        return null

    X = sm.add_constant(s[['time', 'post', 'time_post']])
    y = s[val_col]
    try:
        model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 3})
        null['its_testable'] = True
        null['its_b0'] = model.params.get('const', np.nan)
        null['its_b1'] = model.params.get('time', np.nan)
        null['its_b2'] = model.params.get('post', np.nan)
        null['its_b3'] = model.params.get('time_post', np.nan)
        null['its_b2_se'] = model.bse.get('post', np.nan)
        null['its_b3_se'] = model.bse.get('time_post', np.nan)
        null['its_b2_p']  = model.pvalues.get('post', np.nan)
        null['its_b3_p']  = model.pvalues.get('time_post', np.nan)
    except Exception as ex:
        null['its_testable'] = False
    return null

# Run ITS for each testable event x each proxy (using full series, not windowed)
its_proxies = [
    (c1_m,   'c1_share_active', 'P1_c1_share_active'),
    (c1_m,   'c1_share_total',  'P1_c1_share_total'),
    (sent_m, 'mean_sentiment',  'P2_mean_sentiment'),
    (sent_m, 'pct_positive',    'P2_pct_positive'),
    (sent_m, 'pct_negative',    'P2_pct_negative'),
    (pers_m, 'pct_THE_SUPERFAN','P3_pct_superfan'),
    (pers_m, 'pct_THE_CRITIC',  'P3_pct_critic'),
]

its_rows = []
for _, ev in events_m.iterrows():
    em  = ev['event_month']
    title = ev['Event_Title'].strip()
    for (sdf, vcol, plbl) in its_proxies:
        r = run_its(sdf, em, vcol, plbl, title)
        its_rows.append(r)

its_df = pd.DataFrame(its_rows)
print('ITS rows:', len(its_df))
print('ITS testable:', its_df['its_testable'].sum())

its_ok = its_df[its_df['its_testable']].copy()
print('\n=== TEST 2 ITS RESULTS (testable only) ===')
show = its_ok[['event_title', 'event_month', 'proxy',
               'n_pre_its', 'n_post_its',
               'its_b2', 'its_b2_se', 'its_b2_p',
               'its_b3', 'its_b3_se', 'its_b3_p']].copy()
pd.set_option('display.float_format', '{:.4f}'.format)
print(show.to_string(index=False))

ITS rows: 119
ITS testable: 93

=== TEST 2 ITS RESULTS (testable only) ===
                                                             event_title event_month              proxy  n_pre_its  n_post_its  its_b2  its_b2_se  its_b2_p  its_b3  its_b3_se  its_b3_p
                                     Initial Digital Footprint Expansion     2016-11  P2_mean_sentiment          4         112  0.0260     0.0283    0.3588 -0.0011     0.0108    0.9219
                                     Initial Digital Footprint Expansion     2016-11    P2_pct_positive          4         112  0.0328     0.0285    0.2489 -0.0055     0.0130    0.6709
                                     Initial Digital Footprint Expansion     2016-11    P2_pct_negative          4         112 -0.0046     0.0131    0.7223 -0.0032     0.0055    0.5643
                                     Initial Digital Footprint Expansion     2016-11    P3_pct_superfan         94         112  0.2664     0.0298    0.0000  0.0001     0.0004    0.8749


In [20]:
## Merge Test1 + Test2 and assign tiered verdicts
# Use window=3 for Test1 as the primary window; join on event_month x proxy

t1_w3 = test1_raw[test1_raw['window'] == 3].copy()

merged_results = t1_w3.merge(
    its_df[['event_title', 'event_month', 'proxy',
            'n_pre_its', 'n_post_its',
            'its_b2', 'its_b2_se', 'its_b2_p',
            'its_b3', 'its_b3_se', 'its_b3_p', 'its_testable']],
    on=['event_title', 'event_month', 'proxy'],
    how='left'
)

# Also bring in robustness windows (win=2, win=6) as extra columns
for rwin in [2, 6]:
    rob = test1_raw[test1_raw['window'] == rwin][
        ['event_title', 'event_month', 'proxy', 'fdr_q', 'cliff_delta', 'direction_ok']
    ].rename(columns={
        'fdr_q': f'fdr_q_w{rwin}',
        'cliff_delta': f'cliff_d_w{rwin}',
        'direction_ok': f'dir_ok_w{rwin}'
    })
    merged_results = merged_results.merge(rob, on=['event_title', 'event_month', 'proxy'], how='left')

# Overlap confound: flag at event level (any proxy with overlap is confounded)
merged_results['is_confounded'] = merged_results['p1_overlap_with'].str.len() > 0

def assign_verdict(row):
    """
    Turning point decision rule (from Archive/plans/EVENT_IMPACT_PLAN.md §2):
      supported   : FDR q<0.05 AND |cliff_delta| not negligible AND ITS sig (b2 or b3 p<0.05) AND dir ok AND not confounded
      suggestive  : (q<0.10 OR ITS sig) AND dir ok AND not confounded
      confounded  : confounded (cannot cleanly attribute)
      not_supported: otherwise testable but criteria not met
      untestable  : not testable at all
    Note: ITS tests all events vs the global pre-period, so confound flag mainly applies to P1.
    """
    if not row.get('testable', False):
        return 'untestable'

    q      = row.get('fdr_q', np.nan)
    cd     = row.get('cliff_delta', np.nan)
    cd_mag = row.get('cliff_mag', 'negligible')
    dirok  = row.get('direction_ok', 'ambiguous')
    conf   = row.get('is_confounded', False)
    b2p    = row.get('its_b2_p', np.nan)
    b3p    = row.get('its_b3_p', np.nan)
    its_t  = row.get('its_testable', False)

    its_sig = its_t and (
        (not pd.isna(b2p) and b2p < 0.05) or
        (not pd.isna(b3p) and b3p < 0.05)
    )
    q_sig   = not pd.isna(q) and q < 0.05
    q_sug   = not pd.isna(q) and q < 0.10
    nonneg  = cd_mag not in ('negligible', '')

    if conf and row.get('proxy', '').startswith('P1'):
        return 'confounded'

    if dirok == 'ambiguous':
        # For personal events: just report significance without direction verdict
        if q_sig and its_sig and nonneg:
            return 'suggestive (ambiguous dir)'
        if q_sug or its_sig:
            return 'suggestive (ambiguous dir)'
        return 'not_supported'

    if dirok == 'no':
        return 'not_supported (wrong direction)'

    # dirok == 'yes'
    if q_sig and its_sig and nonneg:
        return 'supported'
    if (q_sug or its_sig) and nonneg:
        return 'suggestive'
    if q_sug or its_sig:
        return 'suggestive (small effect)'
    return 'not_supported'

merged_results['verdict'] = merged_results.apply(assign_verdict, axis=1)

# --- Print overview table ---
print('=== MERGED TEST1+TEST2 RESULTS (window=3 primary) ===')
overview = merged_results[[
    'event_title', 'event_month', 'proxy', 'category',
    'mean_pre', 'mean_post', 'diff_means',
    'cliff_delta', 'cliff_mag', 'fdr_q',
    'its_b2', 'its_b2_p', 'its_b3', 'its_b3_p',
    'direction_ok', 'is_confounded', 'verdict'
]].copy()
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.width', 300)
print(overview.to_string(index=False))

=== MERGED TEST1+TEST2 RESULTS (window=3 primary) ===
                                                             event_title event_month              proxy                    category  mean_pre  mean_post  diff_means  cliff_delta  cliff_mag  fdr_q  its_b2  its_b2_p  its_b3  its_b3_p direction_ok  is_confounded                         verdict
                                     Initial Digital Footprint Expansion     2016-11 P1_c1_share_active      Professional Milestone       NaN        NaN         NaN          NaN               NaN     NaN       NaN     NaN       NaN      no_data          False                      untestable
                                     Initial Digital Footprint Expansion     2016-11  P1_c1_share_total      Professional Milestone       NaN        NaN         NaN          NaN               NaN     NaN       NaN     NaN       NaN      no_data          False                      untestable
                                     Initial Digital Footprint Expansi

In [21]:
## Save full results CSV
merged_results.to_csv(OUTDIR / 'event_test_results.csv', index=False)
print('Saved event_test_results.csv, shape:', merged_results.shape)

## Print verdict summary grouped by event
verdict_summary = (
    merged_results
    .groupby(['event_title', 'event_month', 'category'])['verdict']
    .apply(lambda v: ', '.join(sorted(v.unique())))
    .reset_index()
    .sort_values('event_month')
)
print('\n=== VERDICT SUMMARY BY EVENT ===')
print(verdict_summary.to_string(index=False))

Saved event_test_results.csv, shape: (170, 38)

=== VERDICT SUMMARY BY EVENT ===
                                                             event_title event_month                    category                                                                                           verdict
                                     Initial Digital Footprint Expansion     2016-11      Professional Milestone                            not_supported, not_supported (wrong direction), suggestive, untestable
                                  Rai Radio2 On-Air Broadcast Engagement     2017-09      Professional Milestone                            not_supported, not_supported (wrong direction), suggestive, untestable
                                 Co-Hosting Debut on National Television     2019-09      Professional Milestone                                        not_supported, not_supported (wrong direction), untestable
Macchianera Internet Award & Corporate Social Responsibility Appointment   

In [22]:
## Per-event figures: c1_share_active series + event marker + pre/post means + ITS fit

def its_fitted(series_df, event_m, val_col):
    """Return fitted values from ITS model for plotting."""
    s = series_df.copy()
    if not pd.api.types.is_period_dtype(s['month']):
        s['month'] = pd.PeriodIndex(s['month'].astype(str), freq='M')
    s = s.sort_values('month').reset_index(drop=True)
    s = s[s[val_col].notna()].copy()
    event_ord = event_m.ordinal
    s['time']      = s['month'].apply(lambda m: m.ordinal - event_ord)
    s['post']      = (s['month'] > event_m).astype(int)
    s['time_post'] = s['time'] * s['post']
    if len(s) < 8:
        return s['month'], None
    try:
        X = sm.add_constant(s[['time', 'post', 'time_post']])
        y = s[val_col]
        model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 3})
        s['fitted'] = model.fittedvalues.values
        return s['month'], s['fitted']
    except Exception:
        return s['month'], None

# Events testable on P1 — draw one figure per event
p1_testable_events = (
    merged_results[
        merged_results['proxy'].str.startswith('P1') &
        merged_results['testable']
    ][['event_title', 'event_month']].drop_duplicates()
)

for _, row in p1_testable_events.iterrows():
    title  = row['event_title']
    em_str = row['event_month']
    em     = pd.Period(em_str, freq='M')

    # Shorten title for filename
    slug = em_str + '_' + title[:30].replace(' ', '_').replace("'", '').replace('/', '-')

    fig, ax = plt.subplots(figsize=(12, 4))

    # Full c1_share_active series
    x_months = [m.to_timestamp() for m in c1_m['month']]
    ax.plot(x_months, c1_m['c1_share_active'], color='steelblue', lw=1.5,
            marker='o', ms=3, label='c1_share_active')

    # ITS fitted line
    fit_months, fit_vals = its_fitted(c1_m, em, 'c1_share_active')
    if fit_vals is not None:
        fx = [m.to_timestamp() for m in fit_months]
        ax.plot(fx, fit_vals, color='darkorange', lw=1.5, ls='--', label='ITS fit')

    # Event marker
    ev_ts = em.to_timestamp()
    ax.axvline(ev_ts, color='red', lw=1.5, ls='-', label='Event')

    # Pre/post mean lines (win=3)
    pre_arr, post_arr = get_window(c1_m, em, 3, 'c1_share_active')
    if len(pre_arr) > 0:
        pre_start  = (em - 3).to_timestamp()
        pre_end    = (em - 1).to_timestamp()
        ax.hlines(np.mean(pre_arr),  pre_start,  pre_end,
                  color='green', lw=2, ls='-', label=f'pre mean={np.mean(pre_arr):.3f}')
    if len(post_arr) > 0:
        post_start = (em + 1).to_timestamp()
        post_end   = (em + 3).to_timestamp()
        ax.hlines(np.mean(post_arr), post_start, post_end,
                  color='purple', lw=2, ls='-', label=f'post mean={np.mean(post_arr):.3f}')

    ax.set_title(f'{em_str}  |  {title[:60]}', fontsize=9)
    ax.set_xlabel('Month')
    ax.set_ylabel('C1 share (active denom)')
    ax.legend(fontsize=7, loc='upper left')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.xticks(rotation=45, ha='right', fontsize=7)
    plt.tight_layout()

    fname = OUTDIR / f'fig_p1_{slug}.png'
    fig.savefig(fname, dpi=120)
    plt.close(fig)
    print('Saved:', fname.name)

print('\nAll P1 figures saved.')

Saved: fig_p1_2023-04_Il_saggio_di_fine_anno_First.png


Saved: fig_p1_2024-06_Avanguardia_Pura_Collaborati.png
Saved: fig_p1_2024-06_High-Velocity_Ticket_Sales_Mil.png


Saved: fig_p1_2025-01_Avanguardia_Pura_Winter-Spri.png
Saved: fig_p1_2025-02_Media_Confirmation_of_Relation.png


Saved: fig_p1_2025-03_Autumn_Tour_Circuit_Expansion_.png
Saved: fig_p1_2025-04_Etna_Comics_Marquee_Guest_Book.png


Saved: fig_p1_2025-09_Public_Clarification_of_Interp.png
Saved: fig_p1_2025-10_Autumn_Leg_Re-ignition_of_Ava.png

All P1 figures saved.


## Test 3 — ARIMA Counterfactual (pre-trend extrapolation)

For each candidate event we:
1. Fit ARIMA(p,d,q) on the PRE-period series only (auto-select via AIC over a small grid).
2. Forecast forward into the POST period (point forecast + 95% prediction interval).
3. Compute the **gap** = observed - counterfactual and its cumulative sum.
4. A "supported" signal = observed lies **outside** the 95% PI for ≥ 2 consecutive post-months AND gap direction matches the hypothesis.

Limitation: with no control creator the counterfactual rests on the pre-trend alone.
Seasonal ARIMA is not fitted (only 39 months total); a linear+AR(1) model is used instead.

In [23]:
import itertools
from statsmodels.tsa.arima.model import ARIMA

# ---------------------------------------------------------------------------
# Candidate events for Test 3
# ---------------------------------------------------------------------------
# 2023-04: only 3 P1 pre-months (series starts 2023-01) so P1 is ARIMA-infeasible;
#          P2/P3 have ~92 and ~180 pre-months respectively -> feasible.
# 2024-06: P1 has 17 pre-months, P2/P3 have ~95 and ~197 -> all feasible.
# ---------------------------------------------------------------------------

T3_EVENTS = [
    {
        'event_month': pd.Period('2023-04', freq='M'),
        'label': '2023-04_Solo_Tour',
        'hypothesis_dir': +1,
    },
    {
        'event_month': pd.Period('2024-06', freq='M'),
        'label': '2024-06_Avanguardia_Pura',
        'hypothesis_dir': +1,
    },
]

T3_PROXIES = [
    (c1_m,   'c1_share_active', 'P1_c1_share_active'),
    (sent_m, 'mean_sentiment',  'P2_mean_sentiment'),
    (pers_m, 'pct_THE_SUPERFAN','P3_pct_superfan'),
]

POST_HORIZON = 6
MIN_PRE_ARIMA = 8   # require at least 8 pre-months for a meaningful ARIMA fit

AR_ORDERS = list(itertools.product([0, 1, 2], [0, 1], [0, 1]))

def best_arima(y_pre, orders=AR_ORDERS):
    """Return ARIMA model with lowest AIC over a small grid. Returns None on failure."""
    best_aic   = np.inf
    best_model = None
    for (p, d, q) in orders:
        # Skip orders requiring more params than data
        n_params = p + d + q + 1
        if n_params >= len(y_pre):
            continue
        try:
            m = ARIMA(y_pre, order=(p, d, q), trend='c').fit()
            if m.aic < best_aic:
                best_aic   = m.aic
                best_model = m
        except Exception:
            continue
    return best_model

def run_arima_counterfactual(series_df, event_m, val_col, proxy_label,
                             horizon=POST_HORIZON, min_pre=MIN_PRE_ARIMA):
    """Fit ARIMA on pre-period; forecast forward; compare to observed post."""
    s = series_df.copy()
    if not pd.api.types.is_period_dtype(s['month']):
        s['month'] = pd.PeriodIndex(s['month'].astype(str), freq='M')
    s = s.sort_values('month').reset_index(drop=True)
    s = s[s[val_col].notna()].copy()

    pre  = s[s['month'] < event_m].copy()
    post = s[(s['month'] > event_m) & (s['month'] <= event_m + horizon)].copy()

    result = {
        'proxy': proxy_label,
        'event_month': str(event_m),
        'n_pre': len(pre),
        'n_post_observed': len(post),
        'best_order': None,
        'forecast_months': [],
        'forecast_mean': [],
        'forecast_lo95': [],
        'forecast_hi95': [],
        'observed': [],
        'gap': [],
        'cum_gap': np.nan,
        'n_outside_pi': 0,
        'mean_gap': np.nan,
        'arima_verdict': 'insufficient_data',
    }

    if len(pre) < min_pre or len(post) == 0:
        return result

    y_pre = pre[val_col].values.astype(float)
    model = best_arima(y_pre)
    if model is None:
        result['arima_verdict'] = 'model_failed'
        return result

    result['best_order'] = model.model_orders

    steps     = min(horizon, len(post))
    fc        = model.get_forecast(steps=steps)
    fc_mean   = np.asarray(fc.predicted_mean)

    # conf_int() may return DataFrame or ndarray depending on statsmodels version
    ci = fc.conf_int(alpha=0.05)
    if hasattr(ci, 'values'):
        ci_arr = ci.values          # DataFrame path
    else:
        ci_arr = np.asarray(ci)     # ndarray path
    lo95 = ci_arr[:, 0]
    hi95 = ci_arr[:, 1]

    post_obs    = post[val_col].values[:steps].astype(float)
    post_months = [str(m) for m in post['month'].values[:steps]]
    gap         = post_obs - fc_mean
    n_outside   = int(np.sum((post_obs < lo95) | (post_obs > hi95)))

    result.update({
        'forecast_months': post_months,
        'forecast_mean':   fc_mean.tolist(),
        'forecast_lo95':   lo95.tolist(),
        'forecast_hi95':   hi95.tolist(),
        'observed':        post_obs.tolist(),
        'gap':             gap.tolist(),
        'cum_gap':         float(np.sum(gap)),
        'n_outside_pi':    n_outside,
        'mean_gap':        float(np.mean(gap)),
        'arima_verdict':   'computed',
    })
    return result

# ---------------------------------------------------------------------------
# Run Test 3
# ---------------------------------------------------------------------------
t3_results = []
for ev in T3_EVENTS:
    em    = ev['event_month']
    label = ev['label']
    hyp   = ev['hypothesis_dir']
    print(f'\n--- {label} ---')
    for (sdf, vcol, plbl) in T3_PROXIES:
        r = run_arima_counterfactual(sdf, em, vcol, plbl)
        r['event_label']    = label
        r['hypothesis_dir'] = hyp
        if not np.isnan(r['mean_gap']):
            r['gap_dir_ok'] = 'yes' if np.sign(r['mean_gap']) == hyp else 'no'
        else:
            r['gap_dir_ok'] = 'n/a'
        if r['arima_verdict'] == 'computed':
            if r['n_outside_pi'] >= 2 and r['gap_dir_ok'] == 'yes':
                r['t3_verdict'] = 'supported'
            elif r['n_outside_pi'] >= 1 and r['gap_dir_ok'] == 'yes':
                r['t3_verdict'] = 'suggestive'
            elif r['n_outside_pi'] >= 2:
                r['t3_verdict'] = 'significant_wrong_dir'
            else:
                r['t3_verdict'] = 'within_pi'
        else:
            r['t3_verdict'] = r['arima_verdict']

        order_str = str(r['best_order']) if r['best_order'] else 'N/A'
        print(f'  {plbl}: n_pre={r["n_pre"]}, order={order_str}, '
              f'mean_gap={r["mean_gap"]:.4f}, n_outside={r["n_outside_pi"]}, '
              f'dir={r["gap_dir_ok"]}, verdict={r["t3_verdict"]}')
        t3_results.append(r)

print('\nTest 3 complete.')



--- 2023-04_Solo_Tour ---
  P1_c1_share_active: n_pre=3, order=N/A, mean_gap=nan, n_outside=0, dir=n/a, verdict=insufficient_data


  P2_mean_sentiment: n_pre=80, order={'trend': 0, 'exog': 1, 'ar': 1, 'ma': 0, 'seasonal_ar': 0, 'seasonal_ma': 0, 'reduced_ar': 1, 'reduced_ma': 0, 'exog_variance': 0, 'measurement_variance': 0, 'variance': 1}, mean_gap=0.0427, n_outside=0, dir=yes, verdict=within_pi


  P3_pct_superfan: n_pre=170, order={'trend': 0, 'exog': 1, 'ar': 1, 'ma': 1, 'seasonal_ar': 0, 'seasonal_ma': 0, 'reduced_ar': 1, 'reduced_ma': 1, 'exog_variance': 0, 'measurement_variance': 0, 'variance': 1}, mean_gap=0.0260, n_outside=0, dir=yes, verdict=within_pi

--- 2024-06_Avanguardia_Pura ---


D:\conda_envs\ma_env\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


  P1_c1_share_active: n_pre=17, order={'trend': 0, 'exog': 1, 'ar': 0, 'ma': 0, 'seasonal_ar': 0, 'seasonal_ma': 0, 'reduced_ar': 0, 'reduced_ma': 0, 'exog_variance': 0, 'measurement_variance': 0, 'variance': 1}, mean_gap=-0.0047, n_outside=0, dir=no, verdict=within_pi
  P2_mean_sentiment: n_pre=94, order={'trend': 0, 'exog': 1, 'ar': 0, 'ma': 1, 'seasonal_ar': 0, 'seasonal_ma': 0, 'reduced_ar': 0, 'reduced_ma': 1, 'exog_variance': 0, 'measurement_variance': 0, 'variance': 1}, mean_gap=-0.0482, n_outside=0, dir=no, verdict=within_pi


  P3_pct_superfan: n_pre=184, order={'trend': 0, 'exog': 1, 'ar': 1, 'ma': 1, 'seasonal_ar': 0, 'seasonal_ma': 0, 'reduced_ar': 1, 'reduced_ma': 1, 'exog_variance': 0, 'measurement_variance': 0, 'variance': 1}, mean_gap=-0.0274, n_outside=1, dir=no, verdict=within_pi

Test 3 complete.


In [24]:
## Test 3 figures: one panel per event x proxy

PROXY_TITLES = {
    'P1_c1_share_active': 'C1 share (active denom)',
    'P2_mean_sentiment':  'Mean sentiment score',
    'P3_pct_superfan':    '% THE_SUPERFAN (labelled comments)',
}

for ev in T3_EVENTS:
    em    = ev['event_month']
    label = ev['label']

    ev_results = [r for r in t3_results if r['event_label'] == label and r['arima_verdict'] == 'computed']
    if not ev_results:
        print(f'No computed results for {label}, skipping figure.')
        continue

    n_panels = len(ev_results)
    fig, axes = plt.subplots(n_panels, 1, figsize=(12, 4 * n_panels), sharex=False)
    if n_panels == 1:
        axes = [axes]

    for ax, r in zip(axes, ev_results):
        plbl  = r['proxy']
        vcol  = next(vc for (_, vc, pl) in T3_PROXIES if pl == plbl)
        sdf   = next(sd for (sd, vc, pl) in T3_PROXIES if pl == plbl)

        # Full observed series
        s_full = sdf.copy()
        if not pd.api.types.is_period_dtype(s_full['month']):
            s_full['month'] = pd.PeriodIndex(s_full['month'].astype(str), freq='M')
        s_full = s_full.sort_values('month')
        s_full = s_full[s_full[vcol].notna()]

        x_all = [m.to_timestamp() for m in s_full['month']]
        ax.plot(x_all, s_full[vcol], color='steelblue', lw=1.2, marker='o', ms=3,
                label='Observed', zorder=3)

        # Event vertical line
        ev_ts = em.to_timestamp()
        ax.axvline(ev_ts, color='red', lw=1.5, ls='-', zorder=4, label='Event')

        # ARIMA counterfactual forecast
        fc_months = [pd.Period(m, freq='M').to_timestamp() for m in r['forecast_months']]
        fc_mean   = np.array(r['forecast_mean'])
        fc_lo     = np.array(r['forecast_lo95'])
        fc_hi     = np.array(r['forecast_hi95'])
        obs_post  = np.array(r['observed'])

        ax.plot(fc_months, fc_mean, color='darkorange', lw=1.5, ls='--',
                label='ARIMA counterfactual', zorder=5)
        ax.fill_between(fc_months, fc_lo, fc_hi, color='darkorange', alpha=0.18,
                        label='95% PI')

        # Observed post dots coloured by inside/outside PI
        for xp, yo, lo, hi in zip(fc_months, obs_post, fc_lo, fc_hi):
            color = 'red' if (yo < lo or yo > hi) else 'green'
            ax.scatter([xp], [yo], color=color, s=50, zorder=6)

        verdict_str = r['t3_verdict']
        order_str   = str(r.get('best_order', '?'))
        n_out       = r['n_outside_pi']
        cum         = r['cum_gap']
        ax.set_title(
            f'{label}  |  {PROXY_TITLES.get(plbl, plbl)}\n'
            f'ARIMA order={order_str}  n_pre={r["n_pre"]}  '
            f'n_outside_PI={n_out}  cum_gap={cum:.4f}  verdict={verdict_str}',
            fontsize=8
        )
        ax.set_ylabel(vcol, fontsize=8)
        ax.legend(fontsize=7, loc='upper left')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)

    fig.suptitle(f'Test 3 — ARIMA Counterfactual: {label}', fontsize=10, y=1.01)
    plt.tight_layout()

    fname = OUTDIR / f'fig_t3_{label}.png'
    fig.savefig(fname, dpi=120, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved: {fname.name}')

print('\nAll Test 3 figures saved.')


Saved: fig_t3_2023-04_Solo_Tour.png


Saved: fig_t3_2024-06_Avanguardia_Pura.png

All Test 3 figures saved.


In [25]:
## Save Test 3 results and print summary

# Flatten to CSV (list columns stored as JSON strings for portability)
import json

t3_flat = []
for r in t3_results:
    row = {k: (json.dumps(v) if isinstance(v, list) else v) for k, v in r.items()}
    t3_flat.append(row)

t3_df = pd.DataFrame(t3_flat)
t3_df.to_csv(OUTDIR / 'test3_arima_results.csv', index=False)
print('Saved test3_arima_results.csv, shape:', t3_df.shape)

# Summary table
print('\n=== TEST 3 ARIMA COUNTERFACTUAL SUMMARY ===')
summary_cols = ['event_label', 'proxy', 'n_pre', 'n_post_observed',
                'best_order', 'mean_gap', 'cum_gap', 'n_outside_pi',
                'gap_dir_ok', 't3_verdict']
t3_show = t3_df[summary_cols].copy()
pd.set_option('display.max_colwidth', 35)
pd.set_option('display.width', 250)
pd.set_option('display.float_format', '{:.4f}'.format)
print(t3_show.to_string(index=False))

# Cross-event verdict tally
print('\n=== T3 VERDICT TALLY ===')
print(t3_df.groupby(['event_label', 't3_verdict']).size().to_string())


Saved test3_arima_results.csv, shape: (6, 19)

=== TEST 3 ARIMA COUNTERFACTUAL SUMMARY ===
             event_label              proxy  n_pre  n_post_observed                                                                                                                                                                    best_order  mean_gap  cum_gap  n_outside_pi gap_dir_ok        t3_verdict
       2023-04_Solo_Tour P1_c1_share_active      3                6                                                                                                                                                                          None       NaN      NaN             0        n/a insufficient_data
       2023-04_Solo_Tour  P2_mean_sentiment     80                6 {'trend': 0, 'exog': 1, 'ar': 1, 'ma': 0, 'seasonal_ar': 0, 'seasonal_ma': 0, 'reduced_ar': 1, 'reduced_ma': 0, 'exog_variance': 0, 'measurement_variance': 0, 'variance': 1}    0.0427   0.2564             0        yes         within_